# 🎨 Task 1: Universal Multi-Corruption Denoising Autoencoder

This notebook trains a single universal convolutional autoencoder to reconstruct clean 128x128 RGB images from clean, salt-and-pepper, gaussian blur, or rectangular occlusion inputs on Google Colab (Tesla T4 GPU).

### Pipeline Overview:
1. **Mount Drive & Set Paths**
2. **Sync Repo & Generate/Verify Authentic Manifests from Real Images**
3. **Fast Local NVMe Disk Caching** (copies images once to `/content/local_data/` to eliminate Drive FUSE latency)
4. **Install Dependencies & Configure MLflow SQLite Database**
5. **Hyperparameter Selection (Fixed Optimal or Optuna Study)**
6. **Full 30-Epoch Training of 8x8x128 Bottleneck Architecture** with Cosine Annealing
7. **Comprehensive Fixed-Tier Test Evaluation** (PSNR, SSIM, L1 per severity level)
8. **ONNX Export & Numerical Parity Verification**

### Step 1: Mount Google Drive & Set Paths

In [1]:
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/GenAI-A1'
RAW_OXFORD_DIR = os.path.join(DRIVE_ROOT, 'raw', 'OxfordPet')
OXFORD_IMAGES_DRIVE_DIR = os.path.join(RAW_OXFORD_DIR, 'images_128x128')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
MLRUNS_DIR = os.path.join(DRIVE_ROOT, 'mlruns')
MANIFESTS_DIR = os.path.join(DRIVE_ROOT, 'manifests')
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
EVAL_DIR = os.path.join(DRIVE_ROOT, 'evaluation_task1')

for p in [CHECKPOINTS_DIR, MLRUNS_DIR, MANIFESTS_DIR, EXPORTS_DIR, EVAL_DIR, OXFORD_IMAGES_DRIVE_DIR]:
    os.makedirs(p, exist_ok=True)

print("✅ Google Drive directories configured.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive directories configured.


### Step 2: Clone or Update Repo & Verify Manifests Directly from Real Images
> **Tip:** If re-running after an update, restart the Colab runtime (`Runtime -> Restart Session`) before importing newly pulled Python modules.

In [2]:
import sys
import os
import json

REPO_URL = 'https://github.com/UsmanBari/genai-restoration-studio.git'
REPO_DIR = '/content/genai-restoration-studio'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from data.manifest_generator import create_oxford_pet_manifests

# 1. Scan real image files directly from Google Drive storage
real_files = sorted([
    f for f in os.listdir(OXFORD_IMAGES_DRIVE_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith(('pet_trainval_', 'pet_test_'))
])

print(f"📁 Found {len(real_files)} authentic Oxford-IIIT Pet images in {OXFORD_IMAGES_DRIVE_DIR}")
if len(real_files) == 0:
    raise FileNotFoundError(f"No image files found in {OXFORD_IMAGES_DRIVE_DIR}. Please run Step 4 of colab_bootstrap.ipynb.")

# 2. Construct authentic dataset manifest entries from real files
entries = []
for idx, fname in enumerate(real_files):
    entries.append({
        'image_id': os.path.splitext(fname)[0],
        'filename': fname,
        'split_source': 'trainval' if idx < 3680 else 'test'
    })

# 3. Generate manifests directly into Google Drive MANIFESTS_DIR
create_oxford_pet_manifests(
    image_entries=entries,
    output_dir=MANIFESTS_DIR,
    train_ratio=0.8,
    seed=42,
    img_h=128,
    img_w=128
)

# 4. Rigorous verification check on disk
with open(os.path.join(MANIFESTS_DIR, 'oxford_train_manifest.json'), 'r') as f:
    train_m = json.load(f)
with open(os.path.join(MANIFESTS_DIR, 'oxford_val_manifest.json'), 'r') as f:
    val_m = json.load(f)
with open(os.path.join(MANIFESTS_DIR, 'oxford_test_manifest.json'), 'r') as f:
    test_m = json.load(f)

sample_train = train_m[0]
sample_path = os.path.join(OXFORD_IMAGES_DRIVE_DIR, sample_train['filename'])

print("\n🔍 Manifest Verification Results:")
print(f"  Train samples: {len(train_m)}")
print(f"  Val samples:   {len(val_m)}")
print(f"  Test samples:  {len(test_m)}")
print(f"  Sample entry:  {sample_train}")
print(f"  Sample file '{sample_train['filename']}' exists on disk: {os.path.exists(sample_path)}")

assert os.path.exists(sample_path), f"ERROR: Manifest image {sample_path} not found on disk!"
assert not sample_train['filename'].startswith('pet_trainval_'), "ERROR: Placeholder names detected!"
print("\n✅ Real-image manifests generated and verified successfully!")

/content/genai-restoration-studio
From https://github.com/UsmanBari/genai-restoration-studio
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/genai-restoration-studio
📁 Found 7349 authentic Oxford-IIIT Pet images in /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/images_128x128

🔍 Manifest Verification Results:
  Train samples: 2944
  Val samples:   736
  Test samples:  3669
  Sample entry:  {'image_id': 'basset_hound_150', 'filename': 'basset_hound_150.jpg', 'split_source': 'trainval', 'split': 'train'}
  Sample file 'basset_hound_150.jpg' exists on disk: True

✅ Real-image manifests generated and verified successfully!


### Step 2b: Cache Images to Fast Local Colab Disk
Google Drive's FUSE network filesystem introduces high latency and hangs during random small-file reads. Copying the 128x128 image folder once to local `/content/local_data/` eliminates I/O bottlenecks and speeds up training by 10x-50x.

In [3]:
LOCAL_DATA_DIR = '/content/local_data/OxfordPet'
OXFORD_IMAGES_DIR = os.path.join(LOCAL_DATA_DIR, 'images_128x128')
os.makedirs(OXFORD_IMAGES_DIR, exist_ok=True)

existing_local_count = len([f for f in os.listdir(OXFORD_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

if existing_local_count >= len(real_files):
    print(f"⚡ Local NVMe cache already populated ({existing_local_count} images in {OXFORD_IMAGES_DIR}). Skipping copy.")
else:
    print(f"🚀 Caching {len(real_files)} images from Google Drive to local Colab NVMe disk...")
    !cp -r {OXFORD_IMAGES_DRIVE_DIR}/* {OXFORD_IMAGES_DIR}/
    cached_count = len([f for f in os.listdir(OXFORD_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"✅ Successfully cached {cached_count} images to local path: {OXFORD_IMAGES_DIR}")

print("⚡ Active OXFORD_IMAGES_DIR:", OXFORD_IMAGES_DIR)

⚡ Local NVMe cache already populated (7349 images in /content/local_data/OxfordPet/images_128x128). Skipping copy.
⚡ Active OXFORD_IMAGES_DIR: /content/local_data/OxfordPet/images_128x128


### Step 3: Install Requirements & Configure MLflow SQLite Database

In [4]:
!pip install -q -r requirements-colab.txt

import torch
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

db_path = os.path.join(MLRUNS_DIR, "mlflow.db").replace('\\', '/')
mlflow_uri = f"sqlite:///{db_path}"
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("Task1-Universal-Restoration")

print("✅ PyTorch Version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU Device:", torch.cuda.get_device_name(0))
print(f"✅ MLflow SQLite Tracking URI: {mlflow_uri}")

✅ PyTorch Version: 2.11.0+cu128
✅ CUDA Available: True
✅ GPU Device: Tesla T4
✅ MLflow SQLite Tracking URI: sqlite:////content/drive/MyDrive/GenAI-A1/mlruns/mlflow.db


### Step 4: Hyperparameter Optimization (Optuna Search 2 with Full Parameter Coverage)
Per the assignment requirements, Optuna investigates all core hyperparameters on the **Gated Skip Connection architecture**:
- **Learning Rate (`lr`)**: $10^{-4}$ to $5\times 10^{-3}$ (log-uniform)
- **Batch Size (`batch_size`)**: 16, 32, 64
- **Base Channels (`base_channels`)**: 32, 48, 64
- **Bottleneck Dimension (`bottleneck_dim`)**: 64 (12x compression), 96 (8x compression), 128 (6x compression) — constrained to genuine bottleneck ratios at $8\times 8$ spatial resolution
- **Dropout Rate (`dropout_rate`)**: 0.0, 0.1, 0.2
- **Loss-Weight Value (`alpha`)**: 0.50 to 0.95 (step 0.05)

**Independent Validation Scoring**: Evaluated on $\text{Score}_{\text{val}} = \text{val\_L1} + (1.0 - \text{val\_SSIM})$ to ensure the selected $\alpha$ legitimately balances pixel accuracy and structural quality.

In [5]:
RUN_OPTUNA_SEARCH = True

if RUN_OPTUNA_SEARCH:
    from training.optuna_tuner import run_optuna_study
    print("🔍 Running Optuna Search 2 on Gated Skip Architecture (30 trials, 4 epochs each)...\n")
    study = run_optuna_study(
        manifest_dir=MANIFESTS_DIR,
        images_dir=OXFORD_IMAGES_DIR,
        n_trials=30,
        trial_epochs=4,
        study_name="task1_universal_gated_optuna_search2",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    best_params = dict(study.best_params)
else:
    # Fallback pre-tuned hyperparameters if Optuna search is skipped
    best_params = {
        'lr': 0.000334,
        'batch_size': 32,
        'base_channels': 48,
        'bottleneck_dim': 128,
        'dropout_rate': 0.2,
        'alpha': 0.70
    }

print("\n🚀 Final Training Hyperparameters for Step 5 (Optuna Selected):", best_params)

[I 2026-09-25 23:26:35,607] A new study created in memory with name: task1_universal_gated_optuna_search2


🔍 Running Optuna Search 2 on Gated Skip Architecture (30 trials, 4 epochs each)...

Starting Optuna Hyperparameter Search (30 trials, 4 epochs each)...


/content/genai-restoration-studio/training/optuna_tuner.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.startswith('cuda') else None
/content/genai-restoration-studio/training/trainer_universal.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[I 2026-09-25 23:27:32,241] Trial 0 finished with value: 0.38497549263031583 and parameters: {'lr': 0.0008115387944708458, 'batch_size': 32, 'base_channels': 64, 'bottleneck_dim': 96, 'dropout_rate': 0.0, 'alpha': 0.75}. Best is trial 0 with value: 0.38497549263031583.
/content/genai-restoration-studio/training/optuna_tuner.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if devic


=== Optuna Study Completed ===
Number of finished trials: 30
Best Trial #8:
  Best Validation Score [L1 + (1-SSIM)]: 0.3246
  Best Parameters:
    lr: 0.0002333983594909932
    batch_size: 16
    base_channels: 64
    bottleneck_dim: 96
    dropout_rate: 0.0
    alpha: 0.9

🚀 Final Training Hyperparameters for Step 5 (Optuna Selected): {'lr': 0.0002333983594909932, 'batch_size': 16, 'base_channels': 64, 'bottleneck_dim': 96, 'dropout_rate': 0.0, 'alpha': 0.9}


### Step 5: Full 50-Epoch Training of Gated Skip Autoencoder
Trains the 4-stage convolutional autoencoder with learned gated high-resolution skip connection ($128\times 128 \to 8\times 8\times 128$) for 50 epochs with Cosine Annealing learning rate schedule.

In [6]:
from models.autoencoders import UniversalAutoencoder
from data.oxford_pet import get_oxford_dataloaders
from training.trainer_universal import train_universal_autoencoder

# Dataloaders reading from local high-speed disk with num_workers=0 (prevents Drive FUSE hangs)
batch_size = best_params.get('batch_size', 32)
train_loader, val_loader, test_loader = get_oxford_dataloaders(
    manifest_dir=MANIFESTS_DIR,
    images_dir=OXFORD_IMAGES_DIR,
    batch_size=batch_size,
    num_workers=0
)

# Build Model with Optuna-selected bottleneck dimension and learned gated high-res skip connection
model = UniversalAutoencoder(
    in_channels=3,
    out_channels=3,
    base_channels=best_params.get('base_channels', 48),
    bottleneck_dim=best_params.get('bottleneck_dim', 128),
    dropout_rate=best_params.get('dropout_rate', 0.2)
)

# Full 50-Epoch Training with MLflow run tracking
TOTAL_EPOCHS = 50
with mlflow.start_run(run_name="task1_universal_gated_skip_50epochs") as run:
    mlflow.log_params(best_params)
    mlflow.log_param("total_epochs", TOTAL_EPOCHS)
    mlflow.log_param("bottleneck_spatial", "8x8")
    mlflow.log_param("bottleneck_dim", best_params.get('bottleneck_dim', 128))
    mlflow.log_param("gated_skip", "enabled")

    trained_model, train_res = train_universal_autoencoder(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=TOTAL_EPOCHS,
        lr=best_params.get('lr', 0.000334),
        alpha=best_params.get('alpha', 0.70),
        checkpoints_dir=CHECKPOINTS_DIR,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_mlflow=True
    )

print("\n✅ Full training completed!")
print("Best Checkpoint:", train_res['best_checkpoint_path'])

/content/genai-restoration-studio/training/trainer_universal.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.startswith('cuda') else None


Starting Universal Autoencoder Training on cuda...
  Epochs: 50 | LR: 0.0002333983594909932 | Alpha: 0.9 | Batch Size: 16
Epoch [01/50] (13.9s) Train Loss: 0.1258 (L1: 0.0938, SSIM: 0.587) | Val Loss: 0.1585 (PSNR: 16.02dB, SSIM: 0.492)
  ⭐ New best checkpoint saved (Val Loss: 0.1585)
Epoch [02/50] (12.9s) Train Loss: 0.1022 (L1: 0.0791, SSIM: 0.691) | Val Loss: 0.0902 (PSNR: 20.82dB, SSIM: 0.712)
  ⭐ New best checkpoint saved (Val Loss: 0.0902)
Epoch [03/50] (12.9s) Train Loss: 0.0990 (L1: 0.0760, SSIM: 0.694) | Val Loss: 0.0860 (PSNR: 21.38dB, SSIM: 0.722)
  ⭐ New best checkpoint saved (Val Loss: 0.0860)
Epoch [04/50] (13.0s) Train Loss: 0.0981 (L1: 0.0790, SSIM: 0.730) | Val Loss: 0.0915 (PSNR: 20.60dB, SSIM: 0.732)
Epoch [05/50] (14.0s) Train Loss: 0.0908 (L1: 0.0724, SSIM: 0.744) | Val Loss: 0.0994 (PSNR: 19.67dB, SSIM: 0.742)
Epoch [06/50] (12.7s) Train Loss: 0.0939 (L1: 0.0755, SSIM: 0.741) | Val Loss: 0.1075 (PSNR: 18.65dB, SSIM: 0.739)
Epoch [07/50] (12.8s) Train Loss: 0.0990 

### Step 6: Comprehensive Benchmark Evaluation on Test Manifest
Evaluates on all test images across Clean, S&P, Blur, and Occlusion (low/med/high tiers).

In [7]:
from evaluation.benchmark_universal import run_universal_benchmark

test_manifest_path = os.path.join(MANIFESTS_DIR, 'oxford_test_manifest.json')
eval_results = run_universal_benchmark(
    model=trained_model,
    manifest_path=test_manifest_path,
    images_dir=OXFORD_IMAGES_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=EVAL_DIR,
    num_visualizations=12
)

print("\n=== Benchmark Summary Across Corruption Types ===")
for category, metrics in eval_results['grouped_by_corruption'].items():
    print(f"{category:28s} | PSNR: {metrics['psnr']:5.2f} dB | SSIM: {metrics['ssim']:5.4f} | L1: {metrics['l1']:6.5f}")

print("\n=== Detailed Severity Tier Breakdown ===")
for tier_key, metrics in eval_results['detailed_by_severity_tier'].items():
    print(f"{tier_key:32s} | PSNR: {metrics['psnr']:5.2f} dB | SSIM: {metrics['ssim']:5.4f} | L1: {metrics['l1']:6.5f}")

Generating 12 representative visualization panels...
Benchmark completed! Results saved to /content/drive/MyDrive/GenAI-A1/evaluation_task1/task1_benchmark_results.json

=== Benchmark Summary Across Corruption Types ===
Clean                        | PSNR: 29.20 dB | SSIM: 0.8855 | L1: 0.02647
Salt & Pepper (All Tiers)    | PSNR: 23.95 dB | SSIM: 0.6562 | L1: 0.04568
Gaussian Blur (All Tiers)    | PSNR: 26.20 dB | SSIM: 0.7758 | L1: 0.03700
Occlusion (All Tiers)        | PSNR: 14.98 dB | SSIM: 0.6974 | L1: 0.09488

=== Detailed Severity Tier Breakdown ===
clean_tier_0                     | PSNR: 29.20 dB | SSIM: 0.8855 | L1: 0.02647
salt_and_pepper_tier_1           | PSNR: 26.76 dB | SSIM: 0.7800 | L1: 0.03224
salt_and_pepper_tier_2           | PSNR: 23.82 dB | SSIM: 0.6569 | L1: 0.04429
salt_and_pepper_tier_3           | PSNR: 21.27 dB | SSIM: 0.5318 | L1: 0.06051
gaussian_blur_tier_1             | PSNR: 28.10 dB | SSIM: 0.8570 | L1: 0.02985
gaussian_blur_tier_2             | PSNR: 26

### Step 7: Export Model to ONNX & Verify Numerical Parity

In [8]:
from models.onnx_export import export_to_onnx, verify_onnx_numerical_equivalence

onnx_path = os.path.join(EXPORTS_DIR, "task1_universal.onnx")
export_to_onnx(trained_model, onnx_path)

# Sample test batch for numerical equivalence
sample_batch = next(iter(test_loader))['corrupted'][:8]
parity_res = verify_onnx_numerical_equivalence(trained_model, onnx_path, sample_batch)
print("\nNumerical Verification Result:", parity_res)

Exporting PyTorch model to self-contained ONNX: /content/drive/MyDrive/GenAI-A1/exports/task1_universal.onnx (Opset 16)...
[SUCCESS] ONNX model successfully verified and exported (17.36 MB, 40 weight initializers).

=== ONNX Numerical Parity Verification ===
  PyTorch Output Shape: (8, 3, 128, 128) | Range: [0.0354, 0.9950]
  ONNX Output Shape:    (8, 3, 128, 128) | Range: [0.0354, 0.9950]
  Max Absolute Diff:    4.768372e-07 (Tolerance atol=0.0001)
  Mean Squared Diff:    5.064849e-15
  Parity Status:        PASS (Equivalence Confirmed)

Numerical Verification Result: {'max_abs_diff': 4.76837158203125e-07, 'mean_sq_diff': 5.0648487547001395e-15, 'is_close': True}


In [9]:
!pip install onnxscript -q

### 🎉 Milestone 2 Colab Steps Complete!

**Next Action:** Download/copy the following artifacts from Google Drive (`/content/drive/MyDrive/GenAI-A1/`) to your local workspace:
1. `exports/task1_universal.onnx` $\to` `models/task1_universal.onnx`
2. `evaluation_task1/task1_benchmark_results.json` $\to` `evaluation/task1_benchmark_results.json`
3. `evaluation_task1/figures/` $\to` `evaluation/figures/`
4. Report the cell outputs back to Antigravity so the local backend and frontend can be wired up.